# ExaMon EDA

In this Jupyter Notebook we will query the ExaMon M100 dataset to extract and explore the data available. 

The dataset and query tool are available from https://www.nature.com/articles/s41597-023-02174-3#Sec11.

## Importing the relevant libraries

We begin by importing the libraries necessary for this project. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
from jupyter_dash import JupyterDash
from dash import Dash, dcc, html, Input, Output

## Loading in the data

We will now load in the anomaly detection dataset provided alongside the original dataset.

In [ ]:
# We will store the anomaly detection dataset in 
# the DataFrame anomaly_data
anomaly_dataset_path = "C:/Users/Adam/Documents/Internships/CSD3/ExaMon Data Analysis/exadata/examples/anomaly_detection/388.parquet"
dfAnomalyData = pd.read_parquet(anomaly_dataset_path)

In [ ]:
dfAnomalyData.head(100)

## Preparing the data

Since we are interested in looking at the data related to power consumption, we will filter the dataframe accordingly. 

We will start by creating a list of column names related to power consumption.

In [ ]:
lColumnNames = dfAnomalyData.columns

lPowerColumns = [sColumnName for sColumnName in lColumnNames if "power" in sColumnName]
lPowerColumns.insert(0, "timestamp")

We will now filter the dataframe to contain only the columns related to power consumption.

In [ ]:
dfPowerData = dfAnomalyData.filter(items=lPowerColumns, axis=1)

dfPowerData.head()

We will now allow the user to enter a specified date. We will then filter the dataframe to only show data for the date provided, providing an even smaller dataset to narrow the scope of our analysis.

In [ ]:
sUserDate = input("Please enter a day (between 2020-03-09 and 2022-09-28) in the yyyy-mm-dd format: ")
dtUserDate = np.datetime64(sUserDate).astype(object)

In [ ]:
bDateMask = (dfPowerData["timestamp"].dt.day == dtUserDate.day) & (dfPowerData["timestamp"].dt.month == dtUserDate.month) & (dfPowerData["timestamp"].dt.year == dtUserDate.year)

dfFilteredPower = dfPowerData[bDateMask]

dfFilteredPower

## Exploratory Data Analysis (Matplotlib)

We will now produce some plots and figures to help us explore the dataset.

We will plot a figure of average total power (W) against time.

In [ ]:
# We will start by creating a figure and axes. 
x = np.linspace(0, 97, num=96)
y = dfFilteredPower["total_power_avg"]

len(x)

plt.title(("Average Total Power against Time for " + sUserDate))
plt.ylabel("Average Total Power (W)")
plt.xlabel("Time (24 hour)")

plt.grid()

plt.xticks(ticks=[0, 24, 48, 72, 96], labels=["00:00", "06:00", "12:00", "18:00", "00:00"])
plt.plot(x, y, color='blue')

## Exploratory Data Analysis (Plotly)

We will now recreate the plots above using Plotly to help visualise the differences between the libraries. 

We will plot a figure of average total power (W) against time.

In [ ]:
fig = px.line(dfFilteredPower, x='timestamp', y='total_power_avg')
fig.show()

We will now plot a bar chart of the average total power consumption per month (one for each year of the data).

In [ ]:
lMonths = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

fig = go.Figure()

# For each year we will create a trace bar chart.
for iYear in range(2020, 2023):
    sYear = str(iYear)
    lMonthPower = []
    
    # First we need to calculate the average power for each month of
    # the year. 
    for iMonthCount in range(1,13):
        # We first obtain the date of the required month (so that we 
        # can filter the dataframe)
        if iMonthCount < 10:
            sMonth = "0" + str(iMonthCount)
        else: 
            sMonth  = str(iMonthCount)
        sDate = sYear + "-" + sMonth + "-" + "01"
        dtDate = np.datetime64(sDate).astype(object)

        # We then create a boolean mask and filter the dataframe.
        bDateMask = (dfPowerData["timestamp"].dt.month == dtDate.month) & (dfPowerData["timestamp"].dt.year == dtDate.year)
        dfFilteredMonthPower = dfPowerData[bDateMask]
        
        # We then calculate the average power usage for that month and 
        # store it in the list 'lMonthPower'
        lMonthPower.append(np.mean(dfFilteredMonthPower["total_power_avg"]))
    
    # We then create trace bar charts for each year.
    # We only make the bar chart for the year 2020 visible initially
    # as this is the bar chart we want to show by default. 
    if sYear == "2020":
        fig.add_trace(
        go.Bar(x=lMonths,
           y=lMonthPower,
           name=sYear, 
           visible=True))
    else:
        fig.add_trace(
            go.Bar(x=lMonths,
               y=lMonthPower,
               name=sYear, 
               visible=False))

# Below we create the dropdown menu that allows us to change the trace
# bar chart shown in the figure. 
# The dropdown menu changes which trace bar chart is visible. 
fig.update_layout(
    updatemenus=[
        dict(
            active=0, 
            buttons = list([
                dict(
                    label='2020',
                    method='update',
                    args=[{'visible': [True, False, False]}, 
                         {"title": "Average Power Consumption for 2020"}]),
                dict(
                    label='2021',
                    method='update',
                    args=[{'visible': [False, True, False]}, 
                         {"title": "Average Power Consumption for 2021"}]),
                dict(
                    label='2022',
                    method='update',
                    args=[{'visible': [False, False, True]}, 
                         {"title": "Average Power Consumption for 2022"}])
            ])
        )
    ]
)

# Here we update the layout of the figure, giving the default title. 
fig.update_layout(title=dict(text=("Average Power Consumption for 2020") , font=dict(size=20), automargin=True, yref='paper', x=0.5, pad=dict(b=10)),
    xaxis_title=dict(text="Month", font=dict(size=16, color='black')),
    yaxis_title=dict(text="Average Power (W)", font=dict(size=16, color='black')))

fig.show()

We will now plot a line graph of the average total power consumption per month (one for each year of the data).

In [ ]:
lMonths = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

fig = go.Figure()

# For each year we will create a trace bar chart.
for iYear in range(2020, 2023):
    sYear = str(iYear)
    lMonthPower = []
    
    # First we need to calculate the average power for each month of
    # the year. 
    for iMonthCount in range(1,13):
        # We first obtain the date of the required month (so that we 
        # can filter the dataframe)
        if iMonthCount < 10:
            sMonth = "0" + str(iMonthCount)
        else: 
            sMonth  = str(iMonthCount)
        sDate = sYear + "-" + sMonth + "-" + "01"
        dtDate = np.datetime64(sDate).astype(object)

        # We then create a boolean mask and filter the dataframe.
        bDateMask = (dfPowerData["timestamp"].dt.month == dtDate.month) & (dfPowerData["timestamp"].dt.year == dtDate.year)
        dfFilteredMonthPower = dfPowerData[bDateMask]
        
        # We then calculate the average power usage for that month and 
        # store it in the list 'lMonthPower'
        lMonthPower.append(np.mean(dfFilteredMonthPower["total_power_avg"]))
    
    # We then create trace bar charts for each year.
    # We only make the bar chart for the year 2020 visible initially
    # as this is the bar chart we want to show by default. 
    if sYear == "2020":
        fig.add_trace(
            go.Scatter(x=lMonths,
                   y=lMonthPower,
                   mode='lines+markers',
                   name=sYear,
                   visible=True))
    else:
        fig.add_trace(
            go.Scatter(x=lMonths,
                   y=lMonthPower,
                   mode='lines+markers',
                   name=sYear,
                   visible=False))

# Below we create the dropdown menu that allows us to change the trace
# bar chart shown in the figure. 
# The dropdown menu changes which trace bar chart is visible. 
fig.update_layout(
    updatemenus=[
        dict(
            active=0, 
            buttons = list([
                dict(
                    label='2020',
                    method='update',
                    args=[{'visible': [True, False, False]}, 
                         {"title": "Average Power Consumption for 2020"}]),
                dict(
                    label='2021',
                    method='update',
                    args=[{'visible': [False, True, False]}, 
                         {"title": "Average Power Consumption for 2021"}]),
                dict(
                    label='2022',
                    method='update',
                    args=[{'visible': [False, False, True]}, 
                         {"title": "Average Power Consumption for 2022"}])
            ])
        )
    ]
)

# Here we update the layout of the figure, giving the default title. 
fig.update_layout(title=dict(text=("Average Power Consumption for 2020") , font=dict(size=20), automargin=True, yref='paper', x=0.5, pad=dict(b=10)),
    xaxis_title=dict(text="Month", font=dict(size=16, color='black')),
    yaxis_title=dict(text="Average Power (W)", font=dict(size=16, color='black')))

fig.show()

We will now plot one line graph for each month of the year (with a dropdown menu allowing the year to be selected) all on the same figure. 

In [ ]:
lMonths = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

# Create a grid of subplots
num_rows = 4
num_cols = 3  
fig = sp.make_subplots(rows=num_rows, cols=num_cols,
                      subplot_titles=('January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December'),
                       x_title="timestamp",
                  y_title='total power average (W)'
                      )

# For each year we will create a trace bar chart.
for iYear in range(2020, 2023):
    sYear = str(iYear)
    lMonthPower = []
    
    # First we need to calculate the average power for each month of
    # the year. 
    for iMonthCount in range(1, 13):
        if iMonthCount < 10:
            sMonth = "0" + str(iMonthCount)
        else:
            sMonth = str(iMonthCount)
        sDate = sYear + "-" + sMonth + "-" + "01"
        dtDate = np.datetime64(sDate).astype(object)

        bDateMask = (
            dfPowerData["timestamp"].dt.month == dtDate.month
        ) & (dfPowerData["timestamp"].dt.year == dtDate.year)

        dfFilteredPower = dfPowerData[bDateMask]

        # Add line trace to the corresponding subplot
        row = (iMonthCount - 1) // num_cols + 1
        col = (iMonthCount - 1) % num_cols + 1
        if iYear == 2020:
            fig.add_trace(go.Line(x=dfFilteredPower['timestamp'], y=dfFilteredPower['total_power_avg'], name='Month ' + str(iMonthCount), visible=True), row=row, col=col)
        else:
            fig.add_trace(go.Line(x=dfFilteredPower['timestamp'], y=dfFilteredPower['total_power_avg'], name='Month ' + str(iMonthCount), visible=False), row=row, col=col)

        # Set subplot titles and axis labels
        fig.update_layout(
            title=dict(text=("Average Power Consumption for " + sYear), font=dict(size=20), automargin=True, x=0.5), height=800
        )
    
fig.update_layout(
    updatemenus=[
        dict(
            active=0, 
            buttons = list([
                dict(
                    label='2020',
                    method='update',
                    args=[{'visible': [True, True, True, True, True, True, True, True, True, True, True, True,
                                       False, False, False, False, False, False, False, False, False, False, False, False,
                                       False, False, False, False, False, False, False, False, False, False, False, False]}, 
                         {"title": "Average Daily Power Consumption for Each Month in 2020"}]),
                dict(
                    label='2021',
                    method='update',
                    args=[{'visible': [False, False, False, False, False, False, False, False, False, False, False, False,
                                       True, True, True, True, True, True, True, True, True, True, True, True,
                                       False, False, False, False, False, False, False, False, False, False, False, False]}, 
                         {"title": "Average Daily Power Consumption for Each Month in 2021"}]),
                dict(
                    label='2022',
                    method='update',
                    args=[{'visible': [False, False, False, False, False, False, False, False, False, False, False, False,
                                       False, False, False, False, False, False, False, False, False, False, False, False,
                                       True, True, True, True, True, True, True, True, True, True, True, True]}, 
                         {"title": "Average Daily Power Consumption for Each Month in 2022"}])
            ])
        )
    ]
)

# Here we update the layout of the figure, giving the default title. 
fig.update_layout(title=dict(text=("Average Daily Power Consumption for Each Month in 2020") , font=dict(size=15), automargin=True, yref='paper', x=0.5, pad=dict(b=30)),)

fig.update_layout(showlegend=False)  # Hide individual legends for each subplot

fig.update_xaxes(
    title_standoff = 1000)

fig.show()

We will now plot line graphs of the average total power usage of different components for the date specified earlier in the notebook.

In [ ]:
fig = go.Figure()

# We start by creating trace line graphs for each separate component 
for sColumn in dfFilteredPower.columns:
    if sColumn != 'total_power_avg' and 'avg' in sColumn:
        lComponentData = dfFilteredPower[sColumn]
        lTimeData = dfFilteredPower['timestamp']
        fig.add_trace(
        go.Scatter(x=lTimeData, 
                   y=lComponentData,
                   name=str(sColumn),
                   visible=True))

fig.update_layout(
    title=dict(text=('Average Power Usage for Different Components on ' + sUserDate), font=dict(size=13), automargin=True, yref='paper', x=0.5, pad=dict(b=10)),
    xaxis_title=dict(text="Time (24 hour)", font=dict(size=13, color='black')),
    yaxis_title=dict(text="Average Power (W)", font=dict(size=13, color='black'))
)

fig.show()

We will now plot a histogram (of the daily average total power) in order to visualize the distribution of the power data. 

In [ ]:
lDayPowers = []
lMeans = []

for i in range(len(dfPowerData)):
    if i == 0:
        current_day = dfPowerData.loc[i, 'timestamp'].day
    day = dfPowerData.loc[i, 'timestamp'].day
    if day != current_day:
        mean_power = np.mean(lDayPowers)
        lMeans.append(mean_power)
        if mean_power > 1200:
            interesting_day = dfPowerData.loc[(i - 1), 'timestamp'].day
            interesting_month = dfPowerData.loc[(i - 1), 'timestamp'].month
            interesting_year = dfPowerData.loc[(i - 1), 'timestamp'].year
        lDayPowers = []
        lDayPowers.append(dfPowerData.loc[i, 'total_power_avg'])
        current_day = day
    else:
        lDayPowers.append(dfPowerData.loc[i, 'total_power_avg'])
    
mean_power = np.mean(lDayPowers)
lMeans.append(mean_power)
    
fig = px.histogram(lMeans)
fig.update_layout(showlegend=False,
                  title=dict(text='Histogram of Daily Average Power Usage', x=0.5),
                  yaxis_title=dict(text='Frequency'),
                  xaxis_title=dict(text='Power (W)')
                 )
fig.show()

In [ ]:
bDateMask = (dfAnomalyData["timestamp"].dt.day == interesting_day) & (dfAnomalyData["timestamp"].dt.month == interesting_month) & (dfAnomalyData["timestamp"].dt.year == interesting_year)
dfFilteredInteresting = dfAnomalyData[bDateMask]

dfFilteredInteresting

In [ ]:
dfAnomalyData.set_index('timestamp', inplace=True)

In [ ]:
bMask = ((dfAnomalyData.index >= '2022-06-19') & (dfAnomalyData.index <= '2022-06-21'))

dfInterestingDay = dfAnomalyData[bMask]

In [ ]:
dfInterestingDay.columns.tolist()

In [ ]:
fig = px.line(dfInterestingDay, x=dfInterestingDay.index, y='total_power_avg')
fig.show()

In [ ]:
fig = go.Figure()

# We start by creating trace line graphs for each separate component 
for sColumn in dfFilteredPower.columns:
    if sColumn != 'total_power_avg' and 'avg' in sColumn:
        lComponentData = dfFilteredPower[sColumn]
        lTimeData = dfFilteredPower['timestamp']
        fig.add_trace(
        go.Scatter(x=lTimeData, 
                   y=lComponentData,
                   name=str(sColumn),
                   visible=True))

fig.update_layout(
    title=dict(text=('Average Power Usage for Different Components on ' + sUserDate), font=dict(size=13), automargin=True, yref='paper', x=0.5, pad=dict(b=10)),
    xaxis_title=dict(text="Time (24 hour)", font=dict(size=13, color='black')),
    yaxis_title=dict(text="Average Power (W)", font=dict(size=13, color='black'))
)

fig.show()